# Hour 1 — Setup, Authentication & Exploring Projects

This is the first of five one-hour notebooks that teach the [`dalux_build`](https://pypi.org/project/dalux-build/) Python
client for the **Dalux Build** API. Dalux Build is a construction project management platform (files/BIM, tasks,
forms, inspections, work packages, …); `dalux_build` is a typed Python wrapper around its REST API.

> `dalux_build` is a work in progress. It is not yet feature complete, and the API itself is evolving. This tutorial
> will be updated as the library and API evolve. If you find any issues, please [report them](https://github.com/bruadam/dalux-build/issues)

> `dalux_build` is not affiliated with Dalux. It is an independent open-source project led by [Bruno Adam](https://github.com/bruadam) who is a Dalux customer and user of the API. It is not officially supported by Dalux, but it is used in production by several companies.

**By the end of this hour you will be able to:**

- Configure API credentials with a `.env` file
- Create a `DaluxClient` with `create_client()` and understand what it exposes
- List and inspect **projects**, **companies** and **users**
- Use the `to_dataframe` argument to convert API responses to Pandas DataFrames for analysis

**Roadmap for all three hours:**

| Hour | Topic |
|---|---|
| 1 (this one) | Setup, auth, projects, companies, users, to_dataframe |
| 2 | File areas, folders, files — browse, search, download, upload |
| 3 | Tasks, forms, work packages, and reusable pagination/search patterns |
| 4 | Create graphs and visualizations |
| 5 | Use `dalux_build.webhook_server()` to receive webhooks if a file change or if a set of models is updated once a week |


## 1. Environment check

Make sure the package is installed (it's declared in this repo's `pyproject.toml` and pulled in via `uv sync` / your virtualenv).

In [ ]:
import dalux_build

print(dalux_build.__version__)

## 2. Configure credentials

The client authenticates every request with an `X-API-KEY` header. It reads two environment variables:

- `DALUX_API_KEY` — your company's API key (Dalux Build UI → **Company Profile → Settings → Company overview → API Identities**;
  ask Dalux support to activate API access for your company profile if you don't see this menu) 
  
  > Refer to [Dalux Build API documentation](https://support.dalux.com/hc/en-us/articles/20892369915292-API-identities-in-Dalux-Build-API) for more information on how to generate an API key.

- `DALUX_BASE_URL` — the API base URL for your Dalux node (see `.env.example` in the repo root) for most Dalux Build customers, this is `https://node1.field.dalux.com/service/api/` but some customers are on a different node.

Copy `.env.example` to `.env` in the repo root and fill in your real key **before** running the next cell.
`.env` is already git-ignored, so your key won't be committed.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

# Works whether Jupyter was launched from the repo root or from tutorials/
for candidate in (Path(".env"), Path("../.env")):
    if candidate.exists():
        load_dotenv(candidate)
        break
else:
    load_dotenv()  # fall back to variables already exported in the shell

import os
assert os.getenv("DALUX_API_KEY"), "DALUX_API_KEY not found — copy .env.example to .env and fill it in"
assert os.getenv("DALUX_BASE_URL"), "DALUX_BASE_URL not found — copy .env.example to .env and fill it in"
print("DALUX_BASE_URL:", os.getenv("DALUX_BASE_URL"))

## 3. Create the client — `create_client()`

`create_client()` reads `DALUX_BASE_URL` / `DALUX_API_KEY` from the environment (or accepts them as explicit
`base_url=` / `api_key=` arguments) and returns a `DaluxClient` — a plain dataclass with one attribute per API
resource group. Every attribute is a small class that talks to one part of the Dalux Build API through a shared,
authenticated `requests.Session`.

| Attribute | Class | Covers |
|---|---|---|
| `projects` | `ProjectsApi` | List/get/create/update projects, project metadata |
| `companies` | `CompaniesApi` | Companies on a project |
| `company_catalog` | `CompanyCatalogApi` | Company-profile-wide company catalog |
| `users` | `UsersApi` | Company & project users |
| `file_areas` | `FileAreasApi` | File areas on a project |
| `folders` | `FoldersApi` | Folders within a file area |
| `files` | `FilesApi` | Files within a file area (browse, search, download) |
| `file_upload` | `FileUploadApi` | Chunked upload (create → part → finalize) |
| `file_revisions` | `FileRevisionsApi` | Download historical revision content |
| `tasks` | `TasksApi` | Tasks / issues / approvals / safety observations |
| `forms` | `FormsApi` | Forms and form attachments |
| `work_packages` | `WorkPackagesApi` | Work packages |
| `inspection_plans` | `InspectionPlansApi` | Inspection plans, items, zones, registrations |
| `test_plans` | `TestPlansApi` | Test plans, items, zones, registrations |
| `version_sets` | `VersionSetsApi` | Version sets & their files |
| `project_templates` | `ProjectTemplatesApi` | Project templates on your company profile |

We'll use `projects`, `companies` and `users` this hour; `file_areas`/`folders`/`files`/`file_upload`/`file_revisions`
next hour; and the rest in Hour 3.

In [ ]:
# Create a DaluxClient instance. This is the main entry point to the API.

from dalux_build import create_client

dalux = create_client()
dalux

## 4. Discover your projects — `dalux.projects`

`list_projects()` returns a `ProjectsListResponse` with a typed `.items: List[Project]`. Every response model in
`dalux_build` is a Pydantic model, so `.model_dump()` gives you a plain dict (handy for `pandas`), and fields are
available as real Python attributes (`project.project_name`, not `project["projectName"]`).

### Pretty Print

Before we start exploring projects, let's define a small helper function to pretty-print in order to avoid printing the entire object when we just want to see a few fields.

You can use the another helper function `display()` to display a Pandas DataFrame in a nice table format or just return the object to see the default representation.

In [ ]:
!uv add tabulate -qU

import pandas as pd
from IPython.display import display


def pretty_print(items, title=None):
    """Render a list of dalux_build Pydantic models (Project, Company, User, ...) as a compact table.

    Columns that are None for every item are dropped, since responses like Project
    often come back with several unset optional fields (address, closing, ...).
    """
    items = items if isinstance(items, list) else [items]
    if not items:
        print(f"{title or 'Results'}: none found")
        return

    df = pd.DataFrame([item.model_dump() for item in items]).dropna(axis=1, how="all")
    if title:
        print(f"{title} ({len(df)})")
    print(df.to_markdown(index=False, tablefmt="jupyter"))

In [ ]:
projects_response = dalux.projects.list_projects()
projects_response # This return the list of projects in your account. You can use the first project for the rest of this notebook, or swap this for dalux.projects.

In [ ]:
# In order to get a project by Name, you can use the following.

project_name = "YOUR PROJECT NAME"
project = dalux.projects.get_project_by_name(project_name)
project

In [ ]:
# You can always show the full response object from the API by using the `full_response` parameter. This is useful for debugging or exploring the API.

project_full_response = dalux.projects.list_projects(full_response=True)
project_full_response

In [ ]:
# Notice the difference. You do not get a list of Projects anymore, but also the Meta information about the request.

# Display the Meta information about the request.
project_full_response.metadata

# The metadata constains information about the number of items returned and the number of remaining items before this call. This is useful for paginating through large result sets. -> This is advanced usage and not needed for the rest of these tutorials.

In [ ]:
# Convenience helper: resolve a project ID from its display name
project_id = projects_response[0].project_id
print(project_id)

# Full detail on a single project (GET /5.0/projects/{projectId})
project = dalux.projects.get_project(project_id=project_id)
pretty_print(project.data)

## 5. Companies & users on a project

Most other endpoints in this API are scoped to a single `project_id`, so let's fix one for the rest of the
notebook.

In [ ]:
# Pick the first project in your account to work with for the rest of this notebook.
# Swap this for dalux.projects.get_project_by_name("Your Project Name") if you want a specific one.
projects_response = dalux.projects.list_projects()

PROJECT_ID = projects_response[0].project_id
print("Using project:", projects_response[0].project_name, f"({PROJECT_ID})")

In [ ]:
companies_response = dalux.companies.list_project_companies(project_id=PROJECT_ID)
companies_response

In [ ]:
# Display the companies in a table

companies_response = dalux.companies.list_project_companies(project_id=PROJECT_ID, to_dataframe=True) # Add `to_dataframe=True` to get a Pandas DataFrame instead of a list of Pydantic models.
companies_response

In [ ]:
users_response = dalux.users.list_project_users(project_id=PROJECT_ID, to_dataframe=True)

if users_response.empty:
    print("No users on this project")
else:
    print(f"Users ({len(users_response)})")
    display(users_response)

## 6. More convenient use of a DaluxClient

`DaluxClient` accepts the `project_id` argument, which are passed to every method that requires them. This is convenient if you are working with a single project or file area for the whole notebook.

In [ ]:
new_dalux = create_client(project_id=PROJECT_ID)

project = new_dalux.projects.get_project() # Notice that we don't need to pass project_id here, since it was set when creating the client
pretty_print(project.data, title="Project details")

## 6. Handling errors gracefully

`dalux_build` translates HTTP failures into a small exception hierarchy (all importable from the top-level
package):

- `DaluxError` — base class for everything below
- `NotFoundError` — 404
- `AuthenticationError` — 401 (bad/missing API key)
- `RateLimitError` — 429
- `ApiError` — any other 4xx/5xx
- `ValidationError` — raised *client-side* before a request is even sent (e.g. an empty `project_id`)

Catch `DaluxError` if you just want to handle "anything went wrong with Dalux", or the specific subclass if you
want to react differently (e.g. retry on `RateLimitError`).

## Recap & what's next

You can now:

- Load credentials from `.env` and create an authenticated `DaluxClient`
- List and look up projects, companies and users, and turn results into DataFrames using `to_dataframe=True`
- Set a default `project_id` on the client for convenience
- Catch `dalux_build`'s custom exceptions instead of raw `requests` errors

**Next up — Hour 2:** 
- browsing file areas → folders → files, building a full folder tree, downloading files
(single, filtered-bulk, and by explicit path list), and uploading a file in chunks.
